In [1]:
!pip install kaggle opencv-python torch torchvision tqdm

/bin/bash: /home/solomo/anaconda3/envs/tf/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [2]:
import zipfile
import os

zip_files = ["dataset/nthu-ddd.zip","dataset/mrl-eye.zip"]

for zip_path in zip_files:
    # Define the folder name (e.g., 'yaw-ddd')
    extract_folder = zip_path.replace(".zip", "")
    
    # 1. Check if the destination folder already exists
    if os.path.exists(extract_folder):
        print(f"Directory '{extract_folder}' already exists. Skipping extraction. ✅")
    else:
        # 2. Check if the zip file actually exists before trying to open it
        if os.path.exists(zip_path):
            print(f"Extracting {zip_path} to {extract_folder}...")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_folder)
            print(f"Extraction of {zip_path} Done! ✅")
        else:
            print(f"Warning: {zip_path} not found in the current directory. ")

print("\nAll checks complete.")

Directory 'dataset/nthu-ddd' already exists. Skipping extraction. ✅
Directory 'dataset/mrl-eye' already exists. Skipping extraction. ✅

All checks complete.


In [3]:
import os

def check_structure(root):
    for root, dirs, files in os.walk(root):
        print(root, "->", len(files), "files")

check_structure("dataset/nthu-ddd")
check_structure("dataset/mrl-eye")

dataset/nthu-ddd -> 0 files
dataset/nthu-ddd/Multi class -> 0 files
dataset/nthu-ddd/Multi class/train -> 0 files
dataset/nthu-ddd/Multi class/train/drowsy -> 0 files
dataset/nthu-ddd/Multi class/train/drowsy/yawning -> 8862 files
dataset/nthu-ddd/Multi class/train/drowsy/slowBlinkWithNodding -> 9412 files
dataset/nthu-ddd/Multi class/train/drowsy/sleepyCombination -> 17756 files
dataset/nthu-ddd/Multi class/train/notdrowsy -> 30491 files
dataset/mrl-eye -> 0 files
dataset/mrl-eye/mrleyedataset -> 0 files
dataset/mrl-eye/mrleyedataset/Close-Eyes -> 41946 files
dataset/mrl-eye/mrleyedataset/Open-Eyes -> 42952 files


In [4]:
import os
import shutil

src = "dataset/nthu-ddd/Multi class/train"
dst = "dataset/nthu_processed"

# Define target paths
dst_drowsy = os.path.join(dst, "drowsy")
dst_alert = os.path.join(dst, "alert")

os.makedirs(dst_drowsy, exist_ok=True)
os.makedirs(dst_alert, exist_ok=True)

# --- Process Drowsy ---
# Check if the destination folder is empty before copying
if not os.listdir(dst_drowsy):
    print("Copying drowsy images...")
    drowsy_path = os.path.join(src, "drowsy")
    for sub in os.listdir(drowsy_path):
        sub_path = os.path.join(drowsy_path, sub)
        if os.path.isdir(sub_path):
            for img in os.listdir(sub_path):
                shutil.copy(os.path.join(sub_path, img), dst_drowsy)
else:
    print("Drowsy folder already populated. Skipping.")

# --- Process Alert ---
if not os.listdir(dst_alert):
    print("Copying alert images...")
    alert_path = os.path.join(src, "notdrowsy")
    for img in os.listdir(alert_path):
        shutil.copy(os.path.join(alert_path, img), dst_alert)
else:
    print("Alert folder already populated. Skipping.")

print("NTHU process complete ✅")

Drowsy folder already populated. Skipping.
Alert folder already populated. Skipping.
NTHU process complete ✅


In [5]:
import os

def check_structure(root):
    for root, dirs, files in os.walk(root):
        print(root, "->", len(files), "files")

check_structure("dataset/nthu_processed")
check_structure("dataset/mrl-eye")

dataset/nthu_processed -> 0 files
dataset/nthu_processed/drowsy -> 36030 files
dataset/nthu_processed/alert -> 30491 files
dataset/mrl-eye -> 0 files
dataset/mrl-eye/mrleyedataset -> 0 files
dataset/mrl-eye/mrleyedataset/Close-Eyes -> 41946 files
dataset/mrl-eye/mrleyedataset/Open-Eyes -> 42952 files


In [6]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class SingleFrameDataset(Dataset):
    def __init__(self, drowsy_dirs, alert_dirs, transform=None):
        self.transform = transform
        self.samples = []

        # Load all drowsy/closed-eye images (Label: 1)
        for directory in drowsy_dirs:
            if os.path.exists(directory):
                for img_path in glob.glob(os.path.join(directory, "*.jpg")) + glob.glob(os.path.join(directory, "*.png")):
                    self.samples.append((img_path, 1.0))

        # Load all alert/open-eye images (Label: 0)
        for directory in alert_dirs:
            if os.path.exists(directory):
                for img_path in glob.glob(os.path.join(directory, "*.jpg")) + glob.glob(os.path.join(directory, "*.png")):
                    self.samples.append((img_path, 0.0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor([label], dtype=torch.float32)

# Define your paths based on your notebook output
drowsy_paths = [
    "dataset/mrl-eye/mrleyedataset/Close-Eyes"
]
alert_paths = [
    "dataset/mrl-eye/mrleyedataset/Open-Eyes"
]

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

pretrain_dataset = SingleFrameDataset(drowsy_paths, alert_paths, transform=transform)

# Split 80/20 for CNN Pre-training
train_size = int(0.8 * len(pretrain_dataset))
val_size = len(pretrain_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(pretrain_dataset, [train_size, val_size])

# Using a larger batch size since we aren't loading sequences
pretrain_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Total Pre-training Images: {len(pretrain_dataset)}")

Total Pre-training Images: 84898


In [7]:
import torch.nn as nn
import torchvision.models as models

class SpatialAttention(nn.Module):
    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = torch.cat([avg_out, max_out], dim=1)
        scale = self.conv(scale)
        return x * self.sigmoid(scale)

class PretrainAttentionCNN(nn.Module):
    def __init__(self, feature_dim=512):
        super(PretrainAttentionCNN, self).__init__()
        
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        
        self.attention = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # REMOVE SIGMOID
        self.temp_classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)

        features = x.view(x.size(0), -1)

        return features  #  RETURN FEATURES ONLY

    def classify(self, features):
        return self.temp_classifier(features)

In [8]:
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model = PretrainAttentionCNN().to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.5]).to(device))
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-4)

def train_cnn(model, loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [Pre-train]")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()

            features = model(images)                 #  get features
            outputs = model.classify(features)       #  classify

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            loop.set_postfix(loss=(running_loss/total), acc=(correct/total))
        print(f"Epoch {epoch+1} Train Accuracy: {correct/total:.4f}")
    return model

# Run the pre-training
pretrained_cnn = train_cnn(cnn_model, pretrain_loader, val_loader, epochs=10)

# Save the pre-trained weights so you don't have to do this again
torch.save(pretrained_cnn.state_dict(), "pretrained_attention_cnn.pth")

Epoch 1/10 [Pre-train]: 100%|██████████| 2123/2123 [00:34<00:00, 61.36it/s, acc=0.979, loss=0.00225]


Epoch 1 Train Accuracy: 0.9788


Epoch 2/10 [Pre-train]: 100%|██████████| 2123/2123 [00:32<00:00, 65.70it/s, acc=0.988, loss=0.0012] 


Epoch 2 Train Accuracy: 0.9884


Epoch 3/10 [Pre-train]: 100%|██████████| 2123/2123 [00:30<00:00, 69.08it/s, acc=0.99, loss=0.001]    


...

Epoch 8 Train Accuracy: 0.9948


Epoch 9/10 [Pre-train]: 100%|██████████| 2123/2123 [00:31<00:00, 66.84it/s, acc=0.996, loss=0.000431]


Epoch 9 Train Accuracy: 0.9958


Epoch 10/10 [Pre-train]: 100%|██████████| 2123/2123 [00:31<00:00, 67.29it/s, acc=0.996, loss=0.000404]


Epoch 10 Train Accuracy: 0.9963


In [9]:
import glob
import random

paths = glob.glob("dataset/mrl-eye/mrleyedataset/Close-Eyes/*.png")
img_path = random.choice(paths)

print("Testing image:", img_path)

img = Image.open(img_path).convert("RGB")
img = transform(img)
with torch.no_grad():
    features = cnn_model(img.unsqueeze(0).to(device))
    out = cnn_model.classify(features)
    print(torch.sigmoid(out))

Testing image: dataset/mrl-eye/mrleyedataset/Close-Eyes/s0014_03193_0_1_0_0_0_01.png
tensor([[0.8952]], device='cuda:0')


In [10]:
import glob
import random

paths = glob.glob("dataset/mrl-eye/mrleyedataset/Open-Eyes/*.png")
img_path = random.choice(paths)

print("Testing image:", img_path)

img = Image.open(img_path).convert("RGB")
img = transform(img)
with torch.no_grad():
    features = cnn_model(img.unsqueeze(0).to(device))
    out = cnn_model.classify(features)
    print(torch.sigmoid(out))

Testing image: dataset/mrl-eye/mrleyedataset/Open-Eyes/s0029_01071_0_0_1_0_1_01.png
tensor([[0.0809]], device='cuda:0')


In [11]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class NTHUSequenceDataset(Dataset):
    def __init__(self, root_dir, transform=None, seq_len=8):
        self.transform = transform
        self.seq_len = seq_len
        self.samples = []

        class_map = {
            "alert": 0,
            "drowsy": 1
        }

        for class_name, label in class_map.items():
            class_path = os.path.join(root_dir, class_name)

            if not os.path.exists(class_path):
                continue

            images = sorted(glob.glob(os.path.join(class_path, "*.jpg")))

            # 🔥 IMPORTANT: check count
            if len(images) < seq_len:
                continue

            for i in range(len(images) - seq_len):
                seq = images[i:i+seq_len]
                self.samples.append((seq, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, label = self.samples[idx]

        sequence = []
        for path in frame_paths:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            sequence.append(img)

        seq_tensor = torch.stack(sequence)
        return seq_tensor, torch.tensor(label, dtype=torch.float32)


#  TRANSFORM (IMPORTANT — FACE LEVEL NOW)
transform = transforms.Compose([
    transforms.Resize((112, 112)),   # slightly larger than eye model
    transforms.ToTensor()
])

#  DATASET
nthu_dataset = NTHUSequenceDataset(
    root_dir="dataset/nthu_processed",
    transform=transform,
    seq_len=8   # smaller works better for NTHU
)
print("Total sequences:", len(nthu_dataset))
#  SPLIT
train_size = int(0.8 * len(nthu_dataset))
val_size = len(nthu_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    nthu_dataset, [train_size, val_size]
)

#  LOADERS
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

print(f"NTHU Sequences → Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Total sequences: 66505
NTHU Sequences → Train: 53204, Val: 13301


In [12]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=8):   
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class SequenceDrowsinessModel(nn.Module):
    def __init__(self, pretrained_cnn, seq_length=8, feature_dim=512, num_heads=8, num_layers=2):  
        super(SequenceDrowsinessModel, self).__init__()
        
        self.cnn = pretrained_cnn
        
        
        self.pos_encoder = PositionalEncoding(d_model=feature_dim, max_len=seq_length)
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=feature_dim,
                nhead=num_heads,
                batch_first=True,
                dropout=0.3
            ),
            num_layers=num_layers
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        B, S, C, H, W = x.size()
        
        x = x.view(B * S, C, H, W)
        
        features = self.cnn(x)   # (B*S, 512)
        
        sequence_features = features.view(B, S, -1)  # (B, S, 512)
        
        sequence_features = self.pos_encoder(sequence_features)
        
        transformer_out = self.transformer(sequence_features)
        
        temporal_representation = transformer_out.mean(dim=1)
        
        out = self.classifier(temporal_representation)
        
        return out

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Instantiate CNN
cnn_backbone = PretrainAttentionCNN(feature_dim=512)

# 2. Load YOUR NEW correct weights (IMPORTANT)
cnn_backbone.load_state_dict(torch.load("pretrained_attention_cnn.pth"))

# 3. Move to device
cnn_backbone = cnn_backbone.to(device)

# 4.  PARTIAL FREEZE (REPLACE OLD FREEZE)
for name, param in cnn_backbone.named_parameters():
    if "layer4" in name:
        param.requires_grad = True   # train high-level features
    else:
        param.requires_grad = False  # freeze low-level features

# 5. Initialize final model 
final_model = SequenceDrowsinessModel(
    pretrained_cnn=cnn_backbone,
    seq_length=8   
).to(device)

print("Model successfully built. CNN backbone partially trainable.")

Model successfully built. CNN backbone partially trainable.


In [14]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

#  Loss (stable for logits)
criterion = nn.BCEWithLogitsLoss()

#  Optimizer (good for Transformer + fine-tuning)
optimizer = optim.AdamW(final_model.parameters(), lr=3e-5, weight_decay=1e-3)

#  Scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)


def train_sequence_model(model, train_loader, val_loader, epochs=20):
    for epoch in range(epochs):
        model.train()
        
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for sequences, labels in loop:
            sequences = sequences.to(device)
            labels = labels.unsqueeze(1).to(device)
            
            optimizer.zero_grad()
            
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            
            loss.backward()
            
            #  Prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            #  Accuracy (apply sigmoid manually)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
            
            #  loss calculation
            loop.set_postfix(
                loss=(running_loss / (total_train + 1e-8)),
                acc=(correct_train / total_train)
            )
        
        #  Print epoch training accuracy
        print(f"Epoch {epoch+1} Train Acc: {correct_train / total_train:.4f}")
        
        #  Step scheduler
        scheduler.step()
        
        # ================= VALIDATION =================
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences = sequences.to(device)
                labels = labels.unsqueeze(1).to(device)
                
                outputs = model(sequences)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)
        
        #  Safe validation accuracy
        val_acc = correct_val / total_val if total_val > 0 else 0
        
        print(f"Val Loss: {val_loss / len(val_loader):.4f} | Val Acc: {val_acc:.4f}")
        print("-" * 50)
    
    return model


#  TRAIN
print("Starting robust fine-tuning...")
trained_final_model = train_sequence_model(final_model, train_loader, val_loader, epochs=20)

#  SAVE MODEL
torch.save(trained_final_model.state_dict(), "final_attention_cnn_transformer_v3.pth")

print("✅ Training complete and model saved!")

Starting robust fine-tuning...


Epoch 1/20 [Train]: 100%|██████████| 6651/6651 [06:09<00:00, 18.00it/s, acc=0.721, loss=0.0706]

Epoch 1 Train Acc: 0.7209


Val Loss: 0.4928 | Val Acc: 0.7692
--------------------------------------------------


Epoch 2/20 [Train]: 100%|██████████| 6651/6651 [05:59<00:00, 18.52it/s, acc=0.759, loss=0.0648]

...

Epoch 19 Train Acc: 0.8837


Val Loss: 0.2831 | Val Acc: 0.8950
--------------------------------------------------


Epoch 20/20 [Train]: 100%|██████████| 6651/6651 [05:47<00:00, 19.12it/s, acc=0.886, loss=0.0396]

Epoch 20 Train Acc: 0.8863


Val Loss: 0.3148 | Val Acc: 0.8956
--------------------------------------------------
✅ Training complete and model saved!


In [18]:
import os
import cv2
import torch
from torchvision import transforms
from PIL import Image
from collections import deque
from tqdm import tqdm

# =========================
# 1. Setup Device & Model
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running inference on: {device}")

# match training config
cnn_backbone = PretrainAttentionCNN(feature_dim=512)

# Load CNN weights first
cnn_backbone.load_state_dict(torch.load("pretrained_attention_cnn.pth", map_location=device))

# Build final model (seq_len = 8)
model = SequenceDrowsinessModel(
    pretrained_cnn=cnn_backbone,
    seq_length=8
).to(device)

# Load trained transformer weights
model.load_state_dict(torch.load("final_attention_cnn_transformer_v3.pth", map_location=device))

model.eval()

# =========================
# 2. Transform (MATCH TRAINING)
# =========================
transform = transforms.Compose([
    transforms.Resize((112, 112)),   #  match NTHU training
    transforms.ToTensor()
])

# =========================
# 3. Video Testing Function
# =========================
def test_on_video(input_video_path, output_video_path):

    if not os.path.exists(input_video_path):
        print(f" ERROR: Cannot find input video at '{input_video_path}'")
        return

    cap = cv2.VideoCapture(input_video_path)

    if not cap.isOpened():
        print(" ERROR: Cannot open video")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    os.makedirs(os.path.dirname(output_video_path), exist_ok=True)

    # MJPG works fine even if input is MP4
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    frame_buffer = deque(maxlen=8)   
    prob_history = deque(maxlen=5)   # smoothing

    frame_count = 0

    print(f"✅ Processing video: {input_video_path}")
    loop = tqdm(total=total_frames)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        #  SAMPLE FRAMES (match training)
        if frame_count % 3 == 0:
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)

            tensor_img = transform(pil_img)
            frame_buffer.append(tensor_img)

        #  RUN MODEL WHEN BUFFER FULL
        if len(frame_buffer) == 8:
            sequence_tensor = torch.stack(list(frame_buffer)).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(sequence_tensor)
                prob = torch.sigmoid(output).item()   

            #  SMOOTHING (prevents flicker)
            prob_history.append(prob)
            prob = sum(prob_history) / len(prob_history)

            if prob > 0.5:
                label = f"DROWSY ({prob:.2f})"
                color = (0, 0, 255)
            else:
                label = f"ALERT ({prob:.2f})"
                color = (0, 255, 0)

        else:
            label = "Buffering..."
            color = (255, 255, 0)

        cv2.putText(frame, label, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

        out.write(frame)
        loop.update(1)

    cap.release()
    out.release()
    loop.close()

    print(f"🎉 Done! Output saved to: {output_video_path}")


# =========================
# 4. RUN TEST
# =========================
test_input = "dataset/WhatsApp Video 2026-04-01 at 22.20.15.mp4"
test_output = "dataset/pipeline_test_result_alert.avi"

test_on_video(test_input, test_output)

Running inference on: cuda
✅ Processing video: dataset/WhatsApp Video 2026-04-01 at 22.20.15.mp4


100%|██████████| 237/237 [00:01<00:00, 146.49it/s]

🎉 Done! Output saved to: dataset/pipeline_test_result_alert.avi
